# 03 — Model-specific validation

Validation diagnostics that depend on **the chosen SCM model** (the fit's gap series, weights, RMSPE). For each of the 5 ensemble models from 02 × each event, we report:

| § | Diagnostic | What it tests |
|---|---|---|
| **§5a (i)** | Walk-forward CV (5-fold expanding-window) | Does the model generalize on out-of-sample forecasts across the pre-event window? |
| **§5a (ii)** | Moment matching | Pre-period mean, SD, min, max, AR(1) of log-Brent vs synthetic |
| **§5b** | Parallel-fit defence | Statistics on the pre-period gap series (mean, AR(1), trend slope, R²) |

**No pass/fail thresholds** are applied here, following the report-and-interpret convention of the SCM literature (Abadie 2010/2015/2021) and the Roth (2022) critique of pretests with low statistical power. The diagnostics inform interpretation (drift correction, model-comparison ranking, narrative for the discussion) rather than gatekeeping ensemble inclusion.

## Out of scope here — see [01.5_Donor_Cleanliness.ipynb](01.5_Donor_Cleanliness.ipynb)

Model-**agnostic** tests on the donor pool itself (per-donor SUTVA cleanliness + within-pre-period regime stability) live in 01.5 because they don't depend on the model choice. The model-**native** in-space placebo (§5e (i) in validation.md) lives in `04_Inference.ipynb`.

**Inputs**: saved fits from `data/results/{event}/preferred/shared/{model}/fit.pkl` (produced by `02_Fit_Models.ipynb`).
**Outputs**: validation tables in `data/validation/*.csv`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from lib.config import T0, PRE_WINDOWS, MODEL_HPARAMS, DONOR_POOL_VARIANT
from lib.data import build_panel, load_fit, list_fits, save_validation_table
from lib.validation import (walk_forward_cv, parallel_fit_test, moment_matching,
                            regime_stability_test, event_window_return_test,
                            permutation_mean_shift_test, benjamini_hochberg)

MODELS_AVAILABLE = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bayesian_ridge']
EVENTS = ['russia', 'hormuz']
WINDOW = 'preferred'   # validate the preferred specification only
VARIANT = DONOR_POOL_VARIANT

print(f'Validating event = {EVENTS}, window = {WINDOW}, variant = {VARIANT}')
print(f'Saved fits available: {len(list_fits())}')

Validating event = ['russia', 'hormuz'], window = preferred, variant = shared
Saved fits available: 30


## §5a (i) — Walk-forward CV (5-fold expanding-window)

Splits the pre-period into 5 expanding-window folds (fold $i$ trains on $[t_\text{pre\_start}, \tau_i)$, projects on the next 20 business days). Headline `val_rmse` is the pooled (Hyndman) RMSE over the five non-overlapping fold val residuals; headline `train_rmse` is the cross-fold mean. A model whose val RMSE is much larger than its train RMSE is flagged as overfit — the heuristic `val/train > 2` cutoff is reported but **not** used to auto-exclude models (see [validation.md §5a](../docs/validation.md)).

In [2]:
from lib.validation import get_tuned_hparams

wf_rows = []
wf_folds = []  # per-fold breakdown, saved separately
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS_AVAILABLE:
        # Use the val-tuned hyperparameters that 02_Fit_Models stored with each fit,
        # falling back to config defaults for any keys not tuned (e.g. n_random_v for SCM).
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        try:
            r = walk_forward_cv(model, panel, 'Brent', meta['donors'],
                                t0=meta['t0'], t_pre_start=meta['t_pre_start'], **kwargs)
            # Pop the per-fold DataFrame before flattening to a CSV row.
            # pd.DataFrame on a list of dicts where one value is itself a DataFrame
            # would serialize the str repr — save it as a separate table instead.
            fold_df = r.pop('folds', None)
            if fold_df is not None and len(fold_df) > 0:
                wf_folds.append(fold_df.assign(event=event, window=WINDOW, model=model))
            r.update({'event': event, 'window': WINDOW})
            wf_rows.append(r)
        except Exception as e:
            wf_rows.append({'event': event, 'window': WINDOW, 'model': model,
                            'error': str(e)[:60]})

wf_df = pd.DataFrame(wf_rows)
save_validation_table(wf_df, 'walk_forward_cv')
if wf_folds:
    save_validation_table(pd.concat(wf_folds, ignore_index=True), 'walk_forward_cv_folds')
wf_df.round(4)

,model,train_rmse,val_rmse,val_train_ratio,val_rmse_mean_of_folds,val_rmse_std_of_folds,n_train,n_val,n_folds,horizon,min_train_frac,median_best_iteration,passes,event,window
0,convex_scm,0.1131,0.1412,1.2483,0.1361,0.0422,1520,100,5,20,0.5,NaN,True,russia,preferred
1,ascm,0.0472,0.0922,1.9544,0.0882,0.0299,1520,100,5,20,0.5,NaN,True,russia,preferred
2,elastic_net,0.0516,0.0948,1.8378,0.0902,0.0325,1520,100,5,20,0.5,NaN,True,russia,preferred
3,xgboost,0.0400,0.1247,3.1153,0.1191,0.0413,1520,100,5,20,0.5,1000.0,False,russia,preferred
4,bayesian_ridge,0.0343,0.1068,3.1152,0.0883,0.0672,1520,100,5,20,0.5,NaN,False,russia,preferred
5,convex_scm,0.0597,0.0833,1.3952,0.0764,0.0373,1605,100,5,20,0.5,NaN,True,hormuz,preferred
6,ascm,0.0427,0.0733,1.7172,0.0609,0.0456,1605,100,5,20,0.5,NaN,True,hormuz,preferred
7,elastic_net,0.0485,0.0738,1.5197,0.0583,0.0505,1605,100,5,20,0.5,NaN,True,hormuz,preferred
8,xgboost,0.0447,0.0811,1.8112,0.0679,0.0494,1605,100,5,20,0.5,1000.0,True,hormuz,preferred
9,bayesian_ridge,0.0343,0.0788,2.2979,0.0668,0.0468,1605,100,5,20,0.5,NaN,False,hormuz,preferred


## §5b — Pre-period parallel-fit defence

Diagnostics on each model's pre-period gap series. **Reported and interpreted, not pass/fail tested.**

The `passes` column from `parallel_fit_test()` retains the historical pass/fail logic for backward compatibility but should not be read as a gatekeeper — per validation.md §5b, the SCM literature (Abadie 2010/2015/2021) does not impose formal pre-trend thresholds, and the Roth (2022) "Pretest with caution" critique shows that formal pre-trend tests have low power at typical applied sample sizes.

**How to read the numbers:** large `slope_pct_per_year` magnitudes signal pre-period drift. Use the slope to drift-adjust the post-event gap if needed: `drift_pct = slope × (post-window in years)`. For Russia post-window ≈ 0.58 yr; Hormuz ≈ 0.25 yr.

In [3]:
pf_rows = []
for event in EVENTS:
    for model in MODELS_AVAILABLE:
        fit = load_fit(event, WINDOW, model, variant=VARIANT)
        if fit is None:
            continue
        r = parallel_fit_test(fit)
        r.update({'event': event, 'window': WINDOW, 'model': model})
        pf_rows.append(r)

pf_df = pd.DataFrame(pf_rows)
save_validation_table(pf_df, 'parallel_fit_defence')
pf_df.round(4)

,n,mean_pct,sd_pct,t_stat,p_mean_zero,ar1,slope_pct_per_year,p_slope_zero,r_squared,passes,event,window,model
0,421,0.6148,11.0938,1.1371,0.2562,0.9772,8.9364,0.0000,0.1491,False,russia,preferred,convex_scm
1,421,0.1369,5.2537,0.5347,0.5932,0.8965,0.4732,0.3769,0.0019,True,russia,preferred,ascm
2,421,0.1702,5.8499,0.5970,0.5509,0.9206,1.1790,0.0476,0.0093,False,russia,preferred,elastic_net
3,421,0.0758,3.8591,0.4032,0.6870,0.8245,3.1535,0.0000,0.1534,False,russia,preferred,xgboost
4,421,0.1062,4.6047,0.4731,0.6364,0.8486,0.1753,0.7089,0.0003,True,russia,preferred,bayesian_ridge
5,443,0.2597,6.6969,0.8161,0.4149,0.9566,-5.4267,0.0000,0.1681,False,hormuz,preferred,convex_scm
6,443,0.1198,4.8703,0.5178,0.6048,0.9105,-0.5045,0.2709,0.0027,False,hormuz,preferred,ascm
7,443,0.1363,5.2601,0.5455,0.5857,0.9271,-1.0937,0.0268,0.0111,False,hormuz,preferred,elastic_net
8,443,0.2088,5.3628,0.8196,0.4129,0.9380,-5.9730,0.0000,0.3177,False,hormuz,preferred,xgboost
9,443,0.0682,3.6940,0.3886,0.6978,0.8116,-0.0549,0.8746,0.0001,True,hormuz,preferred,bayesian_ridge


## §5a (ii) — Moment matching

Pre-period mean, SD, min, max, AR(1) of log-Brent vs log-synthetic. Δ Mean ≈ 0 is required; |Δ SD| > 25% of treated SD indicates the donor pool cannot span Brent's volatility.

In [4]:
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS_AVAILABLE:
        fit = load_fit(event, WINDOW, model, variant=VARIANT)
        if fit is None:
            continue
        df = moment_matching(fit, panel, 'Brent')
        df['event'] = event
        df['model'] = model
        save_validation_table(df, f'moments_{event}_{model}')
        print(f'\n{event} / {model}:')
        print(df.round(4).to_string())


russia / convex_scm:
      treated   synth   delta  delta_pct   event       model
mean   4.1281  4.1282  0.0001     0.0015  russia  convex_scm
sd     0.2693  0.2137 -0.0556   -20.6496  russia  convex_scm
min    3.5926  3.7188  0.1261     3.5112  russia  convex_scm
max    4.6216  4.4717 -0.1499    -3.2433  russia  convex_scm
ar1    0.9969  0.9974  0.0006     0.0564  russia  convex_scm

russia / ascm:
      treated   synth   delta  delta_pct   event model
mean   4.1281  4.1281  0.0000     0.0003  russia  ascm
sd     0.2693  0.2623 -0.0070    -2.5963  russia  ascm
min    3.5926  3.6805  0.0878     2.4451  russia  ascm
max    4.6216  4.5540 -0.0676    -1.4628  russia  ascm
ar1    0.9969  0.9977  0.0008     0.0814  russia  ascm

russia / elastic_net:
      treated   synth   delta  delta_pct   event        model
mean   4.1281  4.1281 -0.0000    -0.0000  russia  elastic_net
sd     0.2693  0.2572 -0.0121    -4.4945  russia  elastic_net
min    3.5926  3.6489  0.0563     1.5661  russia  elastic

## Validation summary — numerical only

Per (event, model): walk-forward train/val RMSE + parallel-fit mean and slope + implied drift contribution. **No pass/fail flags** — interpret the numbers narratively (cf. validation.md §5a and §5b). The `wf_passes` / `pf_passes` columns from the legacy strict-threshold rule are retained in the upstream CSV files but should not be read as exclusion criteria.

In [5]:
summary_rows = []
for event in EVENTS:
    post_yr = {'russia': 0.58, 'hormuz': 0.25}.get(event, np.nan)
    for model in MODELS_AVAILABLE:
        wf = wf_df[(wf_df.get('event') == event) & (wf_df.get('model') == model)]
        pf = pf_df[(pf_df.get('event') == event) & (pf_df.get('model') == model)]
        slope = pf['slope_pct_per_year'].iloc[0] if len(pf) else np.nan
        drift_pct = slope * post_yr if not np.isnan(slope) else np.nan
        summary_rows.append({
            'event': event, 'model': model,
            'wf_train_rmse': wf['train_rmse'].iloc[0] if len(wf) else np.nan,
            'wf_val_rmse':   wf['val_rmse'].iloc[0]   if len(wf) else np.nan,
            'wf_ratio':      wf['val_train_ratio'].iloc[0] if len(wf) else np.nan,
            'pf_mean_pct':   pf['mean_pct'].iloc[0] if len(pf) else np.nan,
            'pf_slope_yr':   slope,
            'pf_r2':         pf['r_squared'].iloc[0] if len(pf) else np.nan,
            'drift_contribution_pct': drift_pct,
        })
summary_df = pd.DataFrame(summary_rows)
save_validation_table(summary_df, 'validation_summary')
print('Per (event, model) validation diagnostics:')
print('  wf_*: out-of-sample (walk-forward, model fit on train only)')
print('  pf_*: in-sample pre-period gap series statistics from the saved fit')
print('  drift_contribution_pct = pre-period slope × post-window length (interpretive only)')
print()
print(summary_df.round(4).to_string(index=False))

Per (event, model) validation diagnostics:
  wf_*: out-of-sample (walk-forward, model fit on train only)
  pf_*: in-sample pre-period gap series statistics from the saved fit
  drift_contribution_pct = pre-period slope × post-window length (interpretive only)

 event          model  wf_train_rmse  wf_val_rmse  wf_ratio  pf_mean_pct  pf_slope_yr  pf_r2  drift_contribution_pct
russia     convex_scm         0.1131       0.1412    1.2483       0.6148       8.9364 0.1491                  5.1831
russia           ascm         0.0472       0.0922    1.9544       0.1369       0.4732 0.0019                  0.2745
russia    elastic_net         0.0516       0.0948    1.8378       0.1702       1.1790 0.0093                  0.6838
russia        xgboost         0.0400       0.1247    3.1153       0.0758       3.1535 0.1534                  1.8290
russia bayesian_ridge         0.0343       0.1068    3.1152       0.1062       0.1753 0.0003                  0.1017
hormuz     convex_scm         0.0597 